# Agents, MCP & Skills

En praktisk workshop hvor vi udforsker den nye verden af **agentic software engineering**.

---

## Agenda (ca. 3,5 timer)

| Tid | Emne | Øvelser |
|-----|------|---------|
| 20 min | Intro, Recap & Jupyter Notebooks | |
| 70 min | Agenter - teori og byg din egen | **Øvelse 1:** Tool Execute-funktioner, **Øvelse 2:** Human-in-the-loop |
| 20 min | *Pause* | |
| 45 min | Abstraktionsstigen – MCP | **Øvelse 3:** Higher-level Tools |
| 20 min | Extension: get_nearest_station | **Øvelse 4:** Nyt tool |
| 25 min | Skills – teori og prøv med GitHub Copilot | SKILL.md |
| 10 min | Afrunding | |

---

# Del 0: Intro, Recap & Jupyter Notebooks
*20 minutter*

#### ℹ️ Installér Ollama nu, da det tager lidt tid

**Windows (PowerShell):**
```powershell
winget install Ollama.Ollama
```

**Mac (Homebrew):**
```bash
brew install ollama
```

Start Ollama og hiv modellen ned:
```bash
ollama serve
ollama pull llama3.1:8b
```

👆🏻 kan også foretages i GUI; vælg model, skriv en prompt (f.eks. "hej") og tryk retur

## Opsummering fra sidste gang

- LLMs & Agents - What are they, and why can they code? — Martin
- Agentisk softwareudvikling for personlig produktivitet — Emil
- Agentisk Software Engineering for fuld SDLC af et produkt — Magnus

Lige denne planche er meget relevant mht. dagens seance: [How LLMs Learn: The Three Stages](https://martinrl.github.io/presentations/what-are-llms-and-agents/index.html#14)

<img src="images/LLMs-14.png" alt="LLMs-14" width="600" style="max-width: 100%;">

---

## Intro til Jupyter Notebooks

### Hvad er en Jupyter Notebook?

- `.ipynb` format (JSON-baseret)
- Kører på Python-kernel (ipykernel)
- Understøtter Python, Markdown, og shell-kommandoer
- Industristandard for interaktiv udvikling

### Sådan kører du en celle

- **Shift+Enter** - Kør celle og gå til næste
- **Ctrl+Enter** - Kør celle og bliv
- Variabler persisterer på tværs af celler

In [ ]:
print("Hello EV World!")

---

## Det Agentiske Landskab

<img src="images/Orloff-1600x1067.webp" alt="Professor Orloff" width="600" style="max-width: 100%;">
<br><br>

> An LLM is just a brain in a jar. 🧠

— [Simon Maple](https://www.linkedin.com/in/simonmaple/), Head of Developer Relations @ Tessl ([AI Native podcast](https://open.spotify.com/episode/1HYcUlgkCa9VwvVfuy3fyx?nd=1&dlsi=a77f69fabf8b46b1))

### Hvorfor kan LLM'er kode så godt nu?

**Kode har indbygget feedback.** Til forskel fra "god tekst" kan vi objektivt måle om kode virker:
- Kompilerer det? ✅/❌
- Består tests? ✅/❌
- Løser det opgaven? ✅/❌

Dette muliggør **RLVR** (*Reinforcement Learning with Verifiable Rewards*):

<img src="images/ai-training-loop.svg" alt="AI Træningsloop" width="600" style="max-width: 100%;">

### Benchmarks (pr. januar 2026)

#### SWE-bench: Løs rigtige GitHub issues

**SWE-bench Verified** (500 human-validerede issues):

| # | Model | Score |
|---|-------|-------|
| 1 | Claude Opus 4.5 | 80.9% |
| 2 | GPT-5.2 | 75.4% |
| 3 | Claude Opus 4.1 | 74.5% |

**SWE-bench Pro** (1865 sværere issues, nye repos):

| # | Agent | Score |
|---|-------|-------|
| 1 | Claude Opus 4.5 | 45.9% |
| 2 | Claude Sonnet 4.5 | 43.6% |
| 3 | Gemini 3 Pro Preview | 43.3% |

[swebench.com](https://www.swebench.com/)

---

#### METR: Hvor lang tid kan AI arbejde selvstændigt?

METR måler **tidshorisonten** - hvor lange opgaver kan AI løse med 50% succes?

| # | Model | 50% Tidshorisont |
|---|-------|------------------|
| 1 | Claude Opus 4.5 | ~4.8 timer |
| 2 | GPT-5.1 Codex Max | ~2.9 timer |
| 3 | GPT-5 | ~2.3 timer |

<br><img src="images/metr-timeline.svg" alt="METR Tidslinje" width="600" style="max-width: 100%;">

**Bemærk:**
- Tidshorisonten **fordobles hver 7. måned** (konsistent over 6 år, dvs. en pendant til Moore's Law)

**Hvis trenden fortsætter:**
- 2-4 år: ugelange opgaver
- Årtiet ud: månedlange projekter

[metr.org](https://metr.org/blog/2025-03-19-measuring-ai-ability-to-complete-long-tasks/)

---

#### DPAI Arena: Full developer workflow

JetBrains' benchmark - evaluerer hele engineering lifecycle på Spring-projekter (140+ opgaver).

| # | Agent | Model | Score |
|---|-------|-------|-------|
| 1 | Junie CLI 533.2 | Claude Opus 4.5 | 68.9 |
| 2 | Claude Code 2.0.55 | Claude Opus 4.5 | 68.0 |
| 3 | Junie CLI 496.3-prototype | Claude Sonnet 4.5 | 68.0 |

[dpaia.dev](https://dpaia.dev/)

---

### Men pas på: Benchmark ≠ virkelighed

METR-studie (juli 2025):
- Erfarne udviklere *troede* de var **20% hurtigere** med AI, når objektive tests viste de var **19% langsommere**

Sikkerhedssårbarheder ([Stanford/Boneh 2023](https://arxiv.org/abs/2211.03622)):
- Mennesker uden AI: **~50%** sårbar kode
- Mennesker MED AI: **~64%** — faktisk **værre**!

Forskning tager lang tid og der innoveres med lynets hast, så det handler i høj grad om at etablere et væddemål.

## Prerequisites Verification

### 1. Er Ollama installeret og kørende?

In [ ]:
import urllib.request
import json

selected_model_name = "llama3.1:8b"

try:
    response = urllib.request.urlopen("http://localhost:11434/api/tags")
    print(f"✅ Ollama kører! Model: {selected_model_name}")
except Exception:
    print("❌ Ollama kører IKKE. Start med: ollama serve")

In [ ]:
# Installér ollama Python-pakken
%pip install ollama -q

---

# Del 1: Agenter - teori og byg din egen
*70 minutter inkl. øvelser*

> **"The LLM never actually touches your filesystem. It just *asks* for things to happen."**
> — Mihail Eric, ["The Emperor Has No Clothes"](https://www.mihaileric.com/The-Emperor-Has-No-Clothes/)

AI coding assistants virker som magi, men kernen er simpel:
- **~200 linjer kode** er alt der skal til
- LLM beslutter hvad der skal ske
- **Din kode** udfører handlingerne lokalt
- Resultater sendes tilbage til LLM

### The Agentic Loop - 4 Steps

<img src="images/agentic-loop.svg" alt="Agentic Loop" width="600" style="max-width: 100%;">

**Nøgleindsigt:** LLM'en kalder aldrig noget. Den outputter JSON der siger "Jeg vil gerne kalde X med Y". **Vi** eksekverer det!

## 1.1 Hvad LLM'en Faktisk Ser

Lad os se præcis hvad der sendes til Ollama.

In [ ]:
# Dette er PRÆCIS hvad vi sender til Ollama som tool definition. Det er bare JSON der beskriver en funktion.

tool_json_example = {
    "type": "function",
    "function": {
        "name": "get_station_status",
        "description": "Get real-time status of an EV charging station",
        "parameters": {
            "type": "object",
            "properties": {
                "stationId": {
                    "type": "string",
                    "description": "The station ID, e.g., CPH-001"
                }
            },
            "required": ["stationId"]
        }
    }
}

print("Dette er hvad LLM'en ser som tool definition:")
print(json.dumps(tool_json_example, indent=2))

In [ ]:
# Og dette er hvad LLM'en returnerer når den vil bruge et tool

tool_call_response_example = {
    "message": {
        "role": "assistant",
        "content": "",
        "tool_calls": [
            {
                "function": {
                    "name": "get_station_status",
                    "arguments": {
                        "stationId": "CPH-001"
                    }
                }
            }
        ]
    }
}

print("Når LLM'en vil kalde et tool, returnerer den:")
print(json.dumps(tool_call_response_example, indent=2))
print("\nLLM'en eksekverer ikke noget - den beder os om at gøre det!")

## 1.2 SimpleTool - Den Transparente Abstraktion

Nu bygger vi vores egen tool-definition.

**Forskellen på SimpleTool og ollama-pakkens tool format:**
- `SimpleTool` har en `execute` funktion - du kan se præcis hvad der kører
- `ollama`-pakkens tool format er kun data for JSON serialisering

In [ ]:
from dataclasses import dataclass
from typing import Callable, Any

@dataclass
class SimpleTool:
    name: str
    description: str
    parameter_schema: dict          # JSON schema as dict
    execute: Callable[[dict], str]  # The actual code

print(f"✅ SimpleTool defineret!")

## 1.3 ChargeSmart Domæne - Mock Data

Vi arbejder med et fiktivt EV-ladenetværk i København:

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta

@dataclass
class StationInfo:
    id: str
    name: str
    location: str
    power_kw: int
    type: str
    latitude: float
    longitude: float

stations = {
    "CPH-001": StationInfo("CPH-001", "Nørreport Station", "Nørreport", 150, "ultra-fast", 55.6839, 12.5715),
    "CPH-002": StationInfo("CPH-002", "Fisketorvet Parking", "Fisketorvet", 50, "fast", 55.6692, 12.5519),
    "CPH-003": StationInfo("CPH-003", "Tivoli Garage", "Tivoli", 50, "fast", 55.6736, 12.5681),
    "CPH-004": StationInfo("CPH-004", "Ørestad Center", "Ørestad", 150, "ultra-fast", 55.6310, 12.5770),
    "CPH-005": StationInfo("CPH-005", "Amager Strandpark", "Amager", 22, "slow", 55.6520, 12.6050),
    "CPH-006": StationInfo("CPH-006", "Nørrebro Runddel", "Nørrebro", 50, "fast", 55.7015, 12.5450),
    "CPH-007": StationInfo("CPH-007", "Frederiksberg Have", "Frederiksberg", 7, "slow", 55.6784, 12.5293),
    "CPH-008": StationInfo("CPH-008", "Kastrup Lufthavn P4", "Kastrup", 150, "ultra-fast", 55.6180, 12.6560),
    "CPH-009": StationInfo("CPH-009", "Valby Langgade", "Valby", 22, "slow", 55.6631, 12.5120),
    "CPH-010": StationInfo("CPH-010", "Hellerup Station", "Hellerup", 50, "fast", 55.7280, 12.5720),
}

# Simuler aktive sessioner
active_sessions = {
    "CPH-002": {"vehicle_id": "Tesla-Model-3", "start": datetime.now() - timedelta(minutes=45), "kwh": 28.5},
    "CPH-004": {"vehicle_id": "Polestar-2", "start": datetime.now() - timedelta(minutes=12), "kwh": 18.2},
    "CPH-007": {"vehicle_id": "VW-ID4", "start": datetime.now() - timedelta(hours=2), "kwh": 11.0},
}

# Tarif struktur
def get_tariff(hour: int) -> float:
    if 0 <= hour < 6:
        return 1.50    # Off-peak
    elif 6 <= hour < 17:
        return 2.50    # Normal
    elif 17 <= hour < 21:
        return 4.00    # Peak
    else:
        return 2.50    # Normal

print(f"✅ Loaded {len(stations)} stationer, {len(active_sessions)} aktive sessioner")

## 1.4 Øvelse 1: Forstå Execute-funktionerne til ChargeSmart Tools
*~15 minutter*

Nu skal du **forstå og evt. modificere** `execute`-funktionerne for 3 ChargeSmart tools.

**📋 Opgave:** Før du kører cellen nedenfor:
1. Læs koden igennem og forstå hvad hver funktion gør
2. Prøv at skjule koden og skrive din egen version (brug hints nedenfor)
3. Sammenlign med løsningen og kør cellen

**Du har adgang til:**
- `stations` - Dictionary med alle stationer (ID → StationInfo)
- `active_sessions` - Dictionary med aktive sessioner (stationId → session info)
- `get_tariff(hour)` - Returnerer pris pr. kWh for en given time

**Krav til de 3 tools:**

| Tool | Input | Output |
|------|-------|--------|
| `get_station_status` | stationId (string) | Status på én station (ledig/optaget, kW, type) |
| `list_stations` | *ingen* | Liste over alle stationer med status |
| `calculate_charging_cost` | kwh (number), hour (int) | Pris for at lade X kWh på tidspunkt Y |

In [ ]:
# 🎯 ØVELSE 1: Studer disse Execute-funktioner!
# Tip: args er en dict - brug args["name"] til at hente værdier

def get_station_status_execute(args: dict) -> str:
    station_id = args["stationId"]
    if station_id not in stations:
        return f"Error: Station {station_id} not found"
    station = stations[station_id]
    if station_id in active_sessions:
        session = active_sessions[station_id]
        return (f"Station {station_id} ({station.name}): IN USE by {session['vehicle_id']} "
                f"since {session['start'].strftime('%H:%M')}, {session['kwh']:.1f} kWh delivered. "
                f"{station.power_kw}kW {station.type} charger.")
    return (f"Station {station_id} ({station.name}): AVAILABLE. "
            f"{station.power_kw}kW {station.type} charger at {station.location}.")

def list_stations_execute(args: dict) -> str:
    lines = []
    for s in stations.values():
        status = "IN USE" if s.id in active_sessions else "AVAILABLE"
        lines.append(f"- {s.id}: {s.name} ({s.location}) - {s.power_kw}kW {s.type} - {status}")
    return "ChargeSmart Network Stations:\n" + "\n".join(lines)

def calculate_charging_cost_execute(args: dict) -> str:
    kwh = float(args["kwh"])
    hour = int(args["hour"])
    tariff = get_tariff(hour)
    cost = kwh * tariff
    if 0 <= hour < 6:
        period = "off-peak"
    elif 17 <= hour < 21:
        period = "peak"
    else:
        period = "normal"
    return f"Charging {kwh:.1f} kWh at {hour:02d}:00 ({period} tariff: {tariff:.2f} DKK/kWh) = {cost:.2f} DKK"

tools = {
    "get_station_status": SimpleTool(
        name="get_station_status",
        description="Get real-time status of an EV charging station including availability, power level, and current session info",
        parameter_schema={
            "type": "object",
            "properties": {
                "stationId": {"type": "string", "description": "Station ID like CPH-001"}
            },
            "required": ["stationId"]
        },
        execute=get_station_status_execute
    ),
    "list_stations": SimpleTool(
        name="list_stations",
        description="List all EV charging stations in the ChargeSmart network with their current availability",
        parameter_schema={"type": "object", "properties": {}},
        execute=list_stations_execute
    ),
    "calculate_charging_cost": SimpleTool(
        name="calculate_charging_cost",
        description="Calculate the cost of charging based on kWh needed and time of day",
        parameter_schema={
            "type": "object",
            "properties": {
                "kwh": {"type": "number", "description": "Energy needed in kWh"},
                "hour": {"type": "integer", "description": "Hour of day (0-23)"}
            },
            "required": ["kwh", "hour"]
        },
        execute=calculate_charging_cost_execute
    ),
}

print(f"✅ Defineret {len(tools)} tools:")
for tool in tools.values():
    print(f"   - {tool.name}: {tool.description[:50]}...")

### Hints til Øvelse 1

<details>
<summary>💡 Hint 1: Sådan læser du fra args dict</summary>

```python
# Hent en string property
station_id = args["stationId"]

# Hent et tal (konverter fra mulig string)
kwh = float(args["kwh"])
hour = int(args["hour"])
```

</details>

<details>
<summary>💡 Hint 2: get_station_status struktur</summary>

```python
def get_station_status_execute(args: dict) -> str:
    station_id = args["stationId"]

    if station_id not in stations:
        return f"Error: Station {station_id} not found"

    station = stations[station_id]
    has_session = station_id in active_sessions

    # Returner forskellige beskeder baseret på has_session
    if has_session:
        return f"Station {station_id}: IN USE..."
    else:
        return f"Station {station_id}: AVAILABLE..."
```

</details>

<details>
<summary>💡 Hint 3: list_stations med list comprehension</summary>

```python
def list_stations_execute(args: dict) -> str:
    lines = []
    for s in stations.values():
        status = "IN USE" if s.id in active_sessions else "AVAILABLE"
        lines.append(f"- {s.id}: {s.name} - {status}")
    return "\n".join(lines)
```

</details>

<details>
<summary>✅ Fuld løsning: Alle 3 tools</summary>

```python
def get_station_status_execute(args: dict) -> str:
    station_id = args["stationId"]
    if station_id not in stations:
        return f"Error: Station {station_id} not found"
    station = stations[station_id]
    if station_id in active_sessions:
        session = active_sessions[station_id]
        return (f"Station {station_id} ({station.name}): IN USE by {session['vehicle_id']} "
                f"since {session['start'].strftime('%H:%M')}, {session['kwh']:.1f} kWh delivered. "
                f"{station.power_kw}kW {station.type} charger.")
    return (f"Station {station_id} ({station.name}): AVAILABLE. "
            f"{station.power_kw}kW {station.type} charger at {station.location}.")

def list_stations_execute(args: dict) -> str:
    lines = []
    for s in stations.values():
        status = "IN USE" if s.id in active_sessions else "AVAILABLE"
        lines.append(f"- {s.id}: {s.name} ({s.location}) - {s.power_kw}kW {s.type} - {status}")
    return "ChargeSmart Network Stations:\n" + "\n".join(lines)

def calculate_charging_cost_execute(args: dict) -> str:
    kwh = float(args["kwh"])
    hour = int(args["hour"])
    tariff = get_tariff(hour)
    cost = kwh * tariff
    if 0 <= hour < 6:
        period = "off-peak"
    elif 17 <= hour < 21:
        period = "peak"
    else:
        period = "normal"
    return f"Charging {kwh:.1f} kWh at {hour:02d}:00 ({period} tariff: {tariff:.2f} DKK/kWh) = {cost:.2f} DKK"
```

</details>

### Verificer din løsning

Kør cellen nedenfor for at teste dine Execute-funktioner **uden LLM**:

In [ ]:
# Test dine tools direkte!
print("=== Test get_station_status ===")
print(tools["get_station_status"].execute({"stationId": "CPH-001"}))
print()

print("=== Test list_stations ===")
print(tools["list_stations"].execute({}))
print()

print("=== Test calculate_charging_cost ===")
print(tools["calculate_charging_cost"].execute({"kwh": 50, "hour": 18}))
print()

print("✅ Alle tests kørte! Hvis du ser output ovenfor, virker dine tools.")

## 1.5 The Agentic Loop - ~50 Lines - det er det hele

Nu bygger vi det komplette agentic loop fra bunden. Vi bruger `ollama`-pakken til at kommunikere med Ollama.

In [ ]:
import ollama

print("✅ ollama pakke loaded!")

In [ ]:
# Hjælpefunktion: Konverter SimpleTool til Ollama's tool format
def to_ollama_format(simple_tool: SimpleTool) -> dict:
    return {
        "type": "function",
        "function": {
            "name": simple_tool.name,
            "description": simple_tool.description,
            "parameters": simple_tool.parameter_schema,
        }
    }

print("✅ to_ollama_format helper defined")

In [ ]:
# THE AGENTIC LOOP - ~50 lines, no magic!
def run_agent(user_message: str, tools: dict[str, SimpleTool]) -> str:
    messages = [
        {
            "role": "system",
            "content": """Du er en hjælpsom EV-ladnings assistent for ChargeSmart netværket i København.
                Brug de tilgængelige tools til at besvare spørgsmål om ladestationer,
                tilgængelighed, og priser. Svar altid på dansk."""
        },
        {"role": "user", "content": user_message}
    ]

    ollama_tools = [to_ollama_format(t) for t in tools.values()]
    max_iterations = 10

    for iteration in range(1, max_iterations + 1):
        print(f"\n--- Iteration {iteration} ---")

        # 1. Send til Ollama med tool-definitioner
        response = ollama.chat(
            model=selected_model_name,
            messages=messages,
            tools=ollama_tools,
        )

        assistant_message = response.message
        messages.append(assistant_message.model_dump())

        # 2. Tjek for tool calls
        if not assistant_message.tool_calls:
            # Ingen tool calls = LLM er færdig
            print("✅ LLM finished (no tool calls)")
            return assistant_message.content or "No response"

        # 3. Eksekvér hver tool call SELV
        print(f"🔧 LLM wants to call {len(assistant_message.tool_calls)} tool(s)")

        for tool_call in assistant_message.tool_calls:
            tool_name = tool_call.function.name
            tool_args = tool_call.function.arguments
            print(f"   Tool: {tool_name}")
            print(f"   Args: {tool_args}")

            # Find og kør vores SimpleTool
            if tool_name in tools:
                result = tools[tool_name].execute(tool_args)  # <-- DIN kode kører her!
                print(f"   📤 Result: {result[:80]}...")

                # 4. Tilføj resultat til messages
                messages.append({
                    "role": "tool",
                    "content": result,
                })
            else:
                messages.append({
                    "role": "tool",
                    "content": f"Error: Unknown tool {tool_name}",
                })
        # Loop fortsætter - LLM ser resultaterne

    return "Max iterations reached"

print("✅ run_agent function defined (~50 lines)")

## 1.6 Øvelse 2: Human-in-the-Loop Confirmation
*~15 minutter*

I produktion vil du ofte have tools der gør "farlige" ting - f.eks. starter en ladesession, foretager en betaling, eller sletter data. Før sådanne tools eksekvéres, bør agenten **bede om bekræftelse**.

**Scenarie:** Forestil dig at `calculate_charging_cost` kunne udløse en forhåndsreservation eller pre-authorization på dit kort. Før vi eksekverer det, vil vi gerne have brugerens godkendelse.

**Din opgave:** Modificer loopet så det beder om bekræftelse før "farlige" tools eksekvéres.

In [ ]:
# 🎯 ØVELSE 2: Tilføj human-in-the-loop confirmation

# Definer hvilke tools der kræver bekræftelse
dangerous_tools = {"calculate_charging_cost"}

def run_agent_with_confirmation(user_message: str, tools: dict[str, SimpleTool]) -> str:
    messages = [
        {
            "role": "system",
            "content": """Du er en hjælpsom EV-ladnings assistent for ChargeSmart netværket i København.
                Brug de tilgængelige tools til at besvare spørgsmål om ladestationer,
                tilgængelighed, og priser. Svar altid på dansk."""
        },
        {"role": "user", "content": user_message}
    ]

    ollama_tools = [to_ollama_format(t) for t in tools.values()]
    max_iterations = 10

    for iteration in range(1, max_iterations + 1):
        print(f"\n--- Iteration {iteration} ---")

        response = ollama.chat(
            model=selected_model_name,
            messages=messages,
            tools=ollama_tools,
        )

        assistant_message = response.message
        messages.append(assistant_message.model_dump())

        if not assistant_message.tool_calls:
            print("✅ LLM finished (no tool calls)")
            return assistant_message.content or "No response"

        print(f"🔧 LLM wants to call {len(assistant_message.tool_calls)} tool(s)")

        for tool_call in assistant_message.tool_calls:
            tool_name = tool_call.function.name
            tool_args = tool_call.function.arguments
            print(f"   Tool: {tool_name}")
            print(f"   Args: {tool_args}")

            # 🎯 ØVELSE: Tilføj bekræftelseslogik her!
            # Hvis tool_name er i dangerous_tools:
            #   - Brug input() til at spørge brugeren
            #   - Hvis svaret ikke er "y", skip dette tool call med continue

            if tool_name in tools:
                result = tools[tool_name].execute(tool_args)
                print(f"   📤 Result: {result[:80]}...")

                messages.append({
                    "role": "tool",
                    "content": result,
                })
            else:
                messages.append({
                    "role": "tool",
                    "content": f"Error: Unknown tool {tool_name}",
                })

    return "Max iterations reached"

### Om input()

`input()` er Pythons indbyggede funktion til at bede om brugerinput. I Jupyter vises en inputboks i notebook'en.

<details>
<summary>💡 Reference: Bekræftelseslogikken</summary>

```python
# Tjek om tool er "farligt" og bed om bekræftelse
if tool_name in dangerous_tools:
    confirmation = input(f"⚠️ Execute '{tool_name}'? (y/n): ")
    if confirmation.strip().lower() != "y":
        print("   ⏭️ Skipped by user")
        messages.append({
            "role": "tool",
            "content": f"Tool {tool_name} was skipped by user",
        })
        continue  # Skip til næste tool call
    print("   ✅ Confirmed by user")
```

**Lærdom:** Human-in-the-loop er kritisk for produktions-agents. Du bestemmer hvilke handlinger der kræver godkendelse!

</details>

### Test din løsning

Kør cellen nedenfor. Når LLM vil kalde `calculate_charging_cost`, vil Jupyter vise en input dialog. Skriv `y` for at godkende eller `n` for at afvise.

In [ ]:
# Test human-in-the-loop - Jupyter vil vise en input dialog!
result = run_agent_with_confirmation(
    "Hvad koster det at lade 50 kWh kl. 18?",
    tools
)

print("\n=== Final Response ===")
print(result)

## 1.7 Test din Agent!

In [ ]:
result = run_agent(
    "Hvilke ladestationer er ledige lige nu?",
    tools
)

print("\n=== Final Response ===")
print(result)

In [ ]:
result2 = run_agent(
    "Hvad er status på CPH-002?",
    tools
)

print("\n=== Final Response ===")
print(result2)

In [ ]:
result3 = run_agent(
    "Hvad koster det at lade 50 kWh kl. 18? Og hvad hvis jeg venter til kl. 22?",
    tools
)

print("\n=== Final Response ===")
print(result3)

## Checkpoint Del 1

Du har nu:
1. ✅ Set præcis hvad LLM'en modtager (JSON tool definitions)
2. ✅ Set præcis hvad LLM'en returnerer (JSON tool calls)
3. ✅ Bygget `SimpleTool` - en transparent tool abstraction
4. ✅ **Øvelse 1:** Skrevet Execute-funktioner til 3 ChargeSmart tools
5. ✅ Set det komplette agentic loop (~50 linjer!)
6. ✅ **Øvelse 2:** Tilføjet human-in-the-loop confirmation til loopet
7. ✅ Kørt en agent med dine tools

**Nøgleindsigt:** Det er så simpelt. ~50 linjer loop + ~10 linjer per tool. *The emperor has no clothes!*

**Lærdom fra Øvelse 2:** I produktion er human-in-the-loop kritisk. Du bestemmer hvilke handlinger der kræver godkendelse!

---

# Del 2: Abstraktionsstigen – ollama dict format
*15 minutter (valgfri)*

Nu hvor du forstår hvad der sker "under the hood", lad os se hvordan `ollama`-pakken pakker det samme.

## 2.1 Side-by-Side: SimpleTool vs ollama dict format

In [ ]:
# DIN SimpleTool - alt er synligt:
simple_tool_example = SimpleTool(
    name="get_station_status",
    description="Get EV charging station status",
    parameter_schema={"type": "object", "properties": {"stationId": {"type": "string"}}, "required": ["stationId"]},
    execute=lambda args: f"Station {args['stationId']} is available"  # <-- Koden er HER
)

# ollama-pakkens tool dict - samme data, anden struktur:
ollama_tool_example = {
    "type": "function",
    "function": {
        "name": "get_station_status",
        "description": "Get EV charging station status",
        "parameters": {
            "type": "object",
            "properties": {
                "stationId": {"type": "string", "description": "Station ID"}
            },
            "required": ["stationId"]
        }
    }
}
# BEMÆRK: Ingen execute property! Du skal håndtere eksekveringen separat i dit loop.

print("SimpleTool har:")
print("  - name, description, parameter_schema")
print("  - execute: Callable[[dict], str] \u2190 KODEN ER HER")
print()
print("ollama dict har:")
print("  - type, function (name, description, parameters)")
print("  - Ingen execute! \u2190 Du skal selv matche tool calls til kode")

## 2.2 Nøgleindsigt

**`ollama`-pakkens tool dict er bare en data-struktur til JSON-serialisering.**

Det er derfor vi byggede `SimpleTool` først - for at se det fulde billede:
1. Tool *definition* (JSON schema) → sendes til LLM
2. Tool *execution* (din kode) → kører lokalt når LLM requester det

Libraries som `ollama` giver dig #1, men du skal selv håndtere #2.
Vores `SimpleTool` pakker begge dele sammen så det er tydeligt.

---

# ☕ PAUSE (15 min)

---

# Del 3: Abstraktionsstigen – MCP
*45 minutter inkl. øvelse*

Vi fortsætter op ad abstraktionsstigen: SimpleTool → ollama dict → **MCP**.

## Teori: Model Context Protocol (MCP)

### Problemet: N×M Integrationer

<img src="images/mcp-problem.svg" alt="MCP Problem" width="600" style="max-width: 100%;">

Uden standardisering: Hver agent skal have custom integration til hver tool.

### Løsningen: MCP som Standard

<img src="images/mcp-solution.svg" alt="MCP Solution" width="600" style="max-width: 100%;">

**MCP = USB-C for AI tools.** *Plug any tool into any agent.*

## 3.1 Hvad Ændrer Sig med MCP?

**Før (Del 1-2):** Tools defineret inline i din kode
```python
tools = {
    "get_station_status": SimpleTool(..., execute=get_station_status_execute)
}
# Tools lever i din agent kode
```

**Efter (Del 3):** Tools kommer fra eksterne MCP servere
```python
mcp_client = MCPClient(transport)
tools = mcp_client.list_tools()
# Tools opdages dynamisk fra server!
```

**Samme agent loop, men tools er *"pluggable"*.**

## 3.2 MCP Protocol Basics

MCP bruger **JSON-RPC 2.0** over stdio (lokalt) eller SSE (remote).

Tre hovedoperationer:
1. `initialize` - Handshake mellem client og server
2. `tools/list` - Hent liste af tilgængelige tools
3. `tools/call` - Kald et specifikt tool

## 3.3 Øvelse 3: Higher-Level Tools med ollama-pakken
*~10 minutter*

`ollama`-pakken understøtter automatisk tool-kald via `tools`-parameteren. Vi kan definere tools med Python-funktioner og type hints.

**📋 Opgave:** Sammenlign de nye tool-definitioner med `SimpleTool`:
- I Øvelse 1 brugte du `SimpleTool` med `execute` (dict args)
- Her bruger du standard Python-funktioner med type hints
- Parametrene er **typed** i stedet for rå dicts!

In [ ]:
# 🎯 ØVELSE 3: Definer tools som Python-funktioner

def get_station_status(stationId: str) -> str:
    """Get real-time status of an EV charging station. Parameter: stationId (e.g. CPH-001)"""
    if stationId not in stations:
        return f"Error: Station {stationId} not found"
    station = stations[stationId]
    if stationId in active_sessions:
        session = active_sessions[stationId]
        return (f"Station {stationId} ({station.name}): IN USE by {session['vehicle_id']} "
                f"since {session['start'].strftime('%H:%M')}, {session['kwh']:.1f} kWh delivered. "
                f"{station.power_kw}kW {station.type} charger.")
    return (f"Station {stationId} ({station.name}): AVAILABLE. "
            f"{station.power_kw}kW {station.type} charger at {station.location}.")

def list_stations() -> str:
    """List all EV charging stations in the ChargeSmart network with their current availability"""
    lines = []
    for s in stations.values():
        status = "IN USE" if s.id in active_sessions else "AVAILABLE"
        lines.append(f"- {s.id}: {s.name} ({s.location}) - {s.power_kw}kW {s.type} - {status}")
    return "ChargeSmart Network Stations:\n" + "\n".join(lines)

def calculate_charging_cost(kwh: float, hour: int) -> str:
    """Calculate the cost of charging based on kWh needed and hour of day (0-23)"""
    tariff = get_tariff(hour)
    cost = kwh * tariff
    if 0 <= hour < 6:
        period = "off-peak"
    elif 17 <= hour < 21:
        period = "peak"
    else:
        period = "normal"
    return f"Charging {kwh:.1f} kWh at {hour:02d}:00 ({period} tariff: {tariff:.2f} DKK/kWh) = {cost:.2f} DKK"

ai_tools = [get_station_status, list_stations, calculate_charging_cost]
print(f"🔌 Oprettet {len(ai_tools)} function tools")

### Hints til Øvelse 3

<details>
<summary>💡 Hint: list_stations</summary>

```python
def list_stations() -> str:
    """List all EV charging stations in the ChargeSmart network with their current availability"""
    lines = []
    for s in stations.values():
        status = "IN USE" if s.id in active_sessions else "AVAILABLE"
        lines.append(f"- {s.id}: {s.name} ({s.location}) - {s.power_kw}kW {s.type} - {status}")
    return "ChargeSmart Network Stations:\n" + "\n".join(lines)
```

</details>

<details>
<summary>💡 Hint: calculate_charging_cost</summary>

```python
def calculate_charging_cost(kwh: float, hour: int) -> str:
    """Calculate the cost of charging based on kWh needed and hour of day (0-23)"""
    tariff = get_tariff(hour)
    cost = kwh * tariff
    if 0 <= hour < 6:
        period = "off-peak"
    elif 17 <= hour < 21:
        period = "peak"
    else:
        period = "normal"
    return f"Charging {kwh:.1f} kWh at {hour:02d}:00 ({period} tariff: {tariff:.2f} DKK/kWh) = {cost:.2f} DKK"
```

</details>

<details>
<summary>✅ Fuld løsning: Alle 3 function tools</summary>

```python
def get_station_status(stationId: str) -> str:
    """Get real-time status of an EV charging station. Parameter: stationId (e.g. CPH-001)"""
    if stationId not in stations:
        return f"Error: Station {stationId} not found"
    station = stations[stationId]
    if stationId in active_sessions:
        session = active_sessions[stationId]
        return (f"Station {stationId} ({station.name}): IN USE by {session['vehicle_id']} "
                f"since {session['start'].strftime('%H:%M')}, {session['kwh']:.1f} kWh delivered. "
                f"{station.power_kw}kW {station.type} charger.")
    return (f"Station {stationId} ({station.name}): AVAILABLE. "
            f"{station.power_kw}kW {station.type} charger at {station.location}.")

def list_stations() -> str:
    """List all EV charging stations in the ChargeSmart network with their current availability"""
    lines = []
    for s in stations.values():
        status = "IN USE" if s.id in active_sessions else "AVAILABLE"
        lines.append(f"- {s.id}: {s.name} ({s.location}) - {s.power_kw}kW {s.type} - {status}")
    return "ChargeSmart Network Stations:\n" + "\n".join(lines)

def calculate_charging_cost(kwh: float, hour: int) -> str:
    """Calculate the cost of charging based on kWh needed and hour of day (0-23)"""
    tariff = get_tariff(hour)
    cost = kwh * tariff
    if 0 <= hour < 6:
        period = "off-peak"
    elif 17 <= hour < 21:
        period = "peak"
    else:
        period = "normal"
    return f"Charging {kwh:.1f} kWh at {hour:02d}:00 ({period} tariff: {tariff:.2f} DKK/kWh) = {cost:.2f} DKK"

ai_tools = [get_station_status, list_stations, calculate_charging_cost]
```

</details>

## 3.4 Automatisk Tool-Kald med ollama-pakken

Her kommer automationen af det loop du byggede i Del 1!

`ollama`-pakken kan automatisk konvertere Python-funktioner til tool-definitioner og håndtere tool-kald.

In [ ]:
# Automatisk agentic loop med ollama-pakken
def run_agent_auto(user_message: str, tools_list: list) -> str:
    """Kør agent med automatisk tool invocation via ollama-pakken"""
    messages = [
        {"role": "user", "content": user_message}
    ]

    response = ollama.chat(
        model=selected_model_name,
        messages=messages,
        tools=tools_list,
    )

    # Håndtér tool calls i et loop
    while response.message.tool_calls:
        for tool_call in response.message.tool_calls:
            tool_name = tool_call.function.name
            tool_args = tool_call.function.arguments
            print(f"🔧 Calling: {tool_name}({tool_args})")

            # Find og kald den rigtige funktion
            func = next((t for t in tools_list if t.__name__ == tool_name), None)
            if func:
                result = func(**tool_args)
                print(f"   📤 Result: {result[:80]}...")
                messages.append(response.message.model_dump())
                messages.append({"role": "tool", "content": result})

        response = ollama.chat(
            model=selected_model_name,
            messages=messages,
            tools=tools_list,
        )

    return response.message.content

print("✅ run_agent_auto function defined")

In [ ]:
print("Sender besked med function tools...\n")

result = run_agent_auto(
    "Hvilke ladestationer er ledige lige nu? Og hvad koster det at lade 30 kWh kl. 18?",
    ai_tools
)

print("\n=== Final Response ===")
print(result)

## Checkpoint Del 3

Du har nu:
1. ✅ Forstået MCP som "USB-C for AI tools"
2. ✅ **Øvelse 3:** Oprettet tools med Python-funktioner og type hints
3. ✅ Set hvordan `ollama`-pakken automatiserer loop'et
4. ✅ Kørt en agent med automatiseret tool invocation

**Nøgleindsigt:** MCP standardiserer tool connectivity. I produktion forbinder du til eksterne MCP servere - i notebooks bruger vi Python-funktioner som in-process alternativ.

---

# Del 3.5: Øvelse 4 - Udvid med get_nearest_station
*~20 minutter*

Nu skal du **studere hvordan man udvider** agenten med et helt nyt tool! Dette er den vigtigste øvelse, fordi det simulerer det typiske workflow: du har en fungerende agent og skal tilføje ny funktionalitet.

## Scenarie

En bruger står på gaden og vil finde den **nærmeste ledige ladestation**. De kender deres GPS-koordinater og vil have en anbefaling.

**📋 Opgave:** Studer implementeringen nedenfor og forstå logikken:
1. Modtag brugerens latitude og longitude
2. Filtrer alle **ledige** stationer (ikke optaget)
3. Beregn afstanden til hver station med Haversine
4. Returner den nærmeste

## Stationskoordinater

Alle stationer har nu latitude/longitude (vi tilføjede dem i starten):

| Station | Lokation | Latitude | Longitude |
|---------|----------|----------|-----------|
| CPH-001 | Nørreport | 55.6839 | 12.5715 |
| CPH-002 | Fisketorvet | 55.6692 | 12.5519 |
| CPH-003 | Tivoli | 55.6736 | 12.5681 |
| CPH-004 | Ørestad | 55.6310 | 12.5770 |
| CPH-005 | Amager | 55.6520 | 12.6050 |
| CPH-006 | Nørrebro | 55.7015 | 12.5450 |
| CPH-007 | Frederiksberg | 55.6784 | 12.5293 |
| CPH-008 | Kastrup | 55.6180 | 12.6560 |
| CPH-009 | Valby | 55.6631 | 12.5120 |
| CPH-010 | Hellerup | 55.7280 | 12.5720 |

In [ ]:
import math

# Haversine-formlen til at beregne afstand mellem to koordinater (i km)
def calculate_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    R = 6371  # Jordens radius i km
    d_lat = math.radians(lat2 - lat1)
    d_lon = math.radians(lon2 - lon1)
    a = (math.sin(d_lat / 2) ** 2 +
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) *
         math.sin(d_lon / 2) ** 2)
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

# Test: Afstand fra Rådhuspladsen til Nørreport
test_distance = calculate_distance(55.6761, 12.5683, 55.6839, 12.5715)
print(f"Test: Afstand fra Rådhuspladsen til Nørreport: {test_distance:.2f} km")

# 🎯 DIN TUR: Implementer get_nearest_station
def get_nearest_station(latitude: float, longitude: float) -> str:
    """Find the nearest available charging station. Parameters: latitude and longitude of user's current position"""
    # TODO: 1. Filtrer ledige stationer fra stations.values()
    #       2. Find nærmeste med calculate_distance()
    #       3. Returner beskrivende streng

    raise NotImplementedError("Implementer get_nearest_station!")

ai_tools.append(get_nearest_station)
print(f"🔌 ai_tools har nu {len(ai_tools)} tools (inkl. get_nearest_station stub)")

### Hints til Øvelse 4

<details>
<summary>💡 Hint 1: Filtrer ledige stationer</summary>

```python
available = [s for s in stations.values() if s.id not in active_sessions]

if not available:
    return "Ingen ledige stationer lige nu!"
```

</details>

<details>
<summary>💡 Hint 2: Beregn afstande og find minimum</summary>

```python
nearest = min(available, key=lambda s: calculate_distance(latitude, longitude, s.latitude, s.longitude))
dist = calculate_distance(latitude, longitude, nearest.latitude, nearest.longitude)
```

</details>

<details>
<summary>✅ Fuld løsning</summary>

```python
def get_nearest_station(latitude: float, longitude: float) -> str:
    """Find the nearest available charging station."""
    available = [s for s in stations.values() if s.id not in active_sessions]

    if not available:
        return "Ingen ledige stationer lige nu!"

    nearest = min(available, key=lambda s: calculate_distance(latitude, longitude, s.latitude, s.longitude))
    dist = calculate_distance(latitude, longitude, nearest.latitude, nearest.longitude)

    return (f"Nærmeste ledige station: {nearest.id} ({nearest.name}) - "
            f"{dist:.2f} km væk. "
            f"{nearest.power_kw}kW {nearest.type} charger ved {nearest.location}.")
```

</details>

### Test dit nye tool

In [ ]:
# Test get_nearest_station med en position nær Rådhuspladsen
print("=== Test fra Rådhuspladsen (55.6761, 12.5683) ===\n")

result = run_agent_auto(
    "Jeg står på Rådhuspladsen (koordinater: 55.6761, 12.5683). Hvor er den nærmeste ledige ladestation?",
    ai_tools
)

print("\n=== Final Response ===")
print(result)

## Checkpoint Øvelse 4

Du har nu:
1. ✅ Forstået Haversine-formlen til afstandsberegning
2. ✅ Implementeret `get_nearest_station` med afstandssortering
3. ✅ Udvidet en eksisterende agent med ny funktionalitet
4. ✅ Testet med en realistisk brugerforespørgsel

**Lærdom:** At udvide en agent er ofte nemmere end at bygge en fra bunden. Du tilføjer bare et nyt tool til listen!

---

# Del 4: Skills – teori og prøv med GitHub Copilot
*25 minutter*

## Fra domæne-specifikke agents til universel kode

**Hvordan vi plejede at tænke:**

<img src="images/anthropic-how-we-used-to-think-about-agents.png" alt="Separate agents for each domain" width="600" style="max-width: 100%;">

Vi troede hver domæne kræver sin egen agent med egne tools og scaffolding.

**Hvad vi opdagede:**

<img src="images/anthropic-code-is-the-universal-interface.png" alt="Code is the universal interface" width="600" style="max-width: 100%;">

Kode er ikke bare en use case – det er den universelle grænseflade til den digitale verden.

En coding agent kan:
- Kalde API'er for at hente data (research)
- Organisere data i filsystemet (dokumenter)
- Analysere med Python (finans)
- Generere output i ethvert format (marketing)

**Konklusion**: Vi behøver ikke mange forskellige agents – vi behøver ÉN general-purpose coding agent + **skills** der giver domæneekspertise.

## Teori: Hvorfor Skills?

**Problemet:** Giv en model 100 tools, og den bliver forvirret. Tool selection degraderer.

**Løsningen:** Skills = "lazy-loaded expertise"
- En "router" klassificerer først brugerens intent
- Kun relevante tools/knowledge indlæses
- Modellen fokuserer på én opgave ad gangen

**Key insight:** Copilot har allerede loopet, LLM, og basale tools. Du tilføjer domæneviden!

## 4.1 GitHub Copilot Agent Mode

VS Code 1.108+ har eksperimentel support for "agent skills".

### Aktivering

1. Åbn VS Code Settings (Ctrl+,)
2. Søg efter `chat.useAgentSkills`
3. Aktiver indstillingen

### Skill Struktur

```
.github/skills/
└── ev-charging-advisor/
    └── SKILL.md
```

Copilot scanner automatisk `.github/skills/` og indlæser relevante skills.

## 4.2 SKILL.md Anatomi

```yaml
---
name: my-skill
description: Kort beskrivelse til discovery
triggers:
  - "når bruger spørger om X"
  - "når der er behov for Y"
---

# Detaljerede Instruktioner

Her kommer den fulde domæneviden...
```

- **YAML frontmatter:** Metadata til skill discovery
- **Markdown body:** Instruktioner, eksempler, context

## 4.3 Øvelse: Opret EV Charging Advisor Skill

### Step 1: Opret skill mappe

**PowerShell:**
```powershell
New-Item -ItemType Directory -Force -Path ".github/skills/ev-charging-advisor"
```

**Bash:**
```bash
mkdir -p .github/skills/ev-charging-advisor
```

### Step 2: Opret SKILL.md

Opret filen `.github/skills/ev-charging-advisor/SKILL.md`:

```yaml
---
name: ev-charging-advisor
description: Ekspertrådgivning om optimering af elbil-ladning på ChargeSmart netværket
triggers:
  - "når bruger spørger om ladepris"
  - "når bruger vil finde billigste ladetidspunkt"
  - "når bruger spørger om batteri-tips"
  - "når bruger nævner ChargeSmart eller elbil-ladning"
---

# EV Charging Advisor

Du er en ekspert i at optimere elbil-ladning på ChargeSmart netværket i København.

## ChargeSmart Tarif-struktur

| Periode | Tid | Pris/kWh |
|---------|-----|----------|
| Off-peak | 00:00-06:00 | 1.50 DKK |
| Normal | 06:00-17:00, 21:00-00:00 | 2.50 DKK |
| Peak | 17:00-21:00 | 4.00 DKK |

## Stationer i Netværket

- CPH-001: Nørreport Station - 150kW ultra-fast
- CPH-002: Fisketorvet Parking - 50kW fast
- CPH-003: Tivoli Garage - 50kW fast
- CPH-004: Ørestad Center - 150kW ultra-fast
- CPH-005: Amager Strandpark - 22kW slow
- CPH-006: Nørrebro Runddel - 50kW fast
- CPH-007: Frederiksberg Have - 7kW slow
- CPH-008: Kastrup Lufthavn P4 - 150kW ultra-fast
- CPH-009: Valby Langgade - 22kW slow
- CPH-010: Hellerup Station - 50kW fast

## Instruktioner

Når bruger spørger om ladning:

1. **Identificer behov** - Hvor mange kWh skal de lade?
2. **Tjek tidspunkt** - Hvornår vil de lade?
3. **Beregn pris** - Brug tarif-tabellen
4. **Optimer** - Foreslå ALTID billigere alternativer
5. **Batteri-tips** - Tilføj relevante tips

## Batteri-tips

- Undgå at lade til 100% dagligt (80% er optimalt for battery health)
- Undgå at køre under 20% regelmæssigt
- Forvarm batteri i koldt vejr før hurtigladning
- Ultra-fast ladning (150kW) slider mere på batteriet end langsom ladning

## Eksempel-dialog

**Bruger:** Hvad koster det at lade 40 kWh kl. 18?

**Svar:** At lade 40 kWh kl. 18:00 koster **160 DKK** (peak-tarif: 4.00 DKK/kWh).

💡 **Tip:** Hvis du kan vente til kl. 21:00, falder prisen til **100 DKK** (normal-tarif).
Eller endnu bedre - lad natten over (kl. 00-06) for kun **60 DKK**!

🔋 **Batteri-tip:** Hvis du ikke behøver fuld opladning, så lad kun til 80% for at forlænge batteriets levetid.
```

### Step 3: Test med Copilot Chat

1. Åbn **Copilot Chat** i VS Code (Ctrl+Alt+I)
2. Prøv disse prompts:
   - "Hvad koster det at lade 50 kWh kl. 18?"
   - "Find den billigste tid at lade min elbil"
   - "Giv mig tips til at forlænge mit EV batteri"

Copilot burde nu bruge din skill til at give ChargeSmart-specifikke svar! (Det er tilfældet hvis du ser beskeden "Read SKILL.md file".)

## Checkpoint Del 4

Du har nu:
1. ✅ Forstået Skills som "lazy-loaded expertise"
2. ✅ Oprettet en SKILL.md med domæneviden
3. ✅ Udvidet GitHub Copilot med ChargeSmart ekspertise

**Nøgleindsigt:** Du behøver ikke bygge en agent - du kan udvide én du allerede har!

---

# Afrunding
*10 minutter*

## Key Takeaways

1. **Agents er simple** - ~200 linjer loop kode
2. **"Magien" er i LLM'en** - ikke infrastrukturen
3. **MCP standardiserer** - USB-C for AI tools
4. **Skills tilføjer viden** - uden at genopbygge

## Næste Skridt: Enterprise Agent SDK'er

Nu hvor du har bygget din egen agent-loop og forstår abstraktionerne, kan du udforske etablerede SDK'er. Disse giver dig **observability**, **multi-agent orchestration** og **enterprise compliance** ud af boksen – men bygger på præcis de samme principper, du har lært i dag.

**To tilgange dominerer:**

| | **Anthropic Claude Agent SDK** | **Microsoft Agent Framework** |
|---|---|---|
| **Filosofi** | "Giv Claude en computer" | "Unified multi-agent orchestration" |
| **Kerne** | Bash/fil-tools, subagents, MCP | AutoGen + Semantic Kernel |
| **Sprog** | Python | Python, C#, Java |
| **Styrke** | Simpel agent → computer-interaktion | Enterprise: observability, compliance, A2A |
| **MCP** | ✅ Native integration | ✅ Native integration |

![Agent SDK Landscape](images/agent-sdk-landscape.svg)

> **Nøgleindsigt:** Begge SDK'er bygger på de samme principper, du har lært (tools, loops, MCP). Forskellen ligger i *hvor meget* infrastruktur du får "gratis" – og *hvilken leverandør* du binder dig til.

## The Emperor Has No Clothes

> **"The core of these tools isn't magic. It's about 200 lines of straightforward Python."**
> — Mihail Eric

Du har nu set det selv. Du kan bygge en agent. Du forstår hvad der sker.

**Hvorfor er dette vigtigt?**

> Whilst software development/programming is now dead. We however deeply need software engineers with these skills who understand that LLMs are a new form of programmable computer. If you haven't built your own coding agent yet - please do.
> — [Geoffrey Huntley](https://ghuntley.com/loop/)

## Stof til eftertanke

Kan du komme i tanke om systemer hvor statiske formularer, der forudsætter, at brugeren på forhånd ved, hvad vedkommende har brug for, kunne erstattes af multi-modale brugergrænseflader, der forstår brugerens intention?

![Gammel vs. Ny UX-Logik](images/gammel-vs-ny-logik-ux.png)

## Ressourcer

- [MCP Documentation](https://modelcontextprotocol.io)
- [ollama Python package](https://github.com/ollama/ollama-python)
- [GitHub Copilot Skills](https://docs.github.com/en/copilot)
- [The Emperor Has No Clothes](https://www.mihaileric.com/The-Emperor-Has-No-Clothes/)
- [Ollama](https://ollama.com/)
- [Building agents with the Claude Agent SDK](https://www.anthropic.com/engineering/building-agents-with-the-claude-agent-sdk)
- [Introducing Microsoft Agent Framework](https://azure.microsoft.com/en-us/blog/introducing-microsoft-agent-framework/)

---